# 教師あり学習と教師なし学習

機械学習の入口で最初に分かれるのが、正解ラベルがあるかどうかです。同じ「データから何かを学ぶ」でも、正解があるなら予測問題になり、正解がないなら構造を見つける問題になります。

このノートでは、その違いを概念だけでなくコードの流れで比べます。分類、クラスタリング、PCA を順に見て、「どんな問いにどの枠組みを使うのか」を整理します。

## まずは、問いの種類が違うことを押さえる

教師あり学習は「正解に近づく」ための学習です。教師なし学習は「正解なしでデータのまとまりや軸を見つける」ための学習です。アルゴリズム名より先に、この問いの違いを押さえる方がずっと重要です。


このノートでは、最初にラベル付きデータで分類を行い、次にラベルなしデータでクラスタリングを行い、最後に PCA でデータを圧縮して眺めます。

同じ数値データでも、何を知りたいかで評価の仕方も出力の読み方も変わることが分かるはずです。


## このノートで使う基本語彙

- 教師あり学習: ラベルを使って予測器を作る学習
- 教師なし学習: ラベルなしで構造を探す学習
- クラスタリング: 似たものどうしをまとめる操作
- 次元圧縮: 情報をなるべく保ったまま少数の軸へ写す操作

大事なのは、「精度が高いか」だけで全部を比べようとしないことです。教師なし学習では、そもそも正解率という見方がそのまま使えないことも多いです。


## 同じデータでも、見る指標が変わる

分類では accuracy や混同行列を見ますが、クラスタリングでは inertia や silhouette を見ます。PCA では explained variance ratio を見ます。

どの数字を見るかは、学習の目的に従って変わります。


## 先に構えたい誤解

教師なし学習は「正解がないから評価できない」のではなく、「分類と同じ基準では評価しにくい」が正確です。逆に、教師あり学習では accuracy だけを見ていると、どのクラスを取りこぼしているかを見逃しやすくなります。


## どこまでをこのノートで扱うか

ここでは、分類、クラスタリング、PCA の入口に絞ります。クラスタ数の厳密な選び方や次元圧縮の理論は、ここでは深入りしません。


## まずは、教師あり学習を予測問題として見る

最初にラベル付きデータを使って分類を行います。ここで重要なのは、モデルが `0/1` のラベルを目標として学ぶことです。正解があるからこそ、予測の当たり外れを数えられます。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification, make_blobs, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, silhouette_score

sns.set_theme(style="whitegrid", context="notebook")


## 教師あり学習: ラベルありデータで予測する

まずは二値分類データを作り、`0` と `1` を予測する問題を作ります。ここでは「ラベルがある」ことが核心で、モデルはそのラベルを目標に学習します。

In [ ]:
X, y = make_classification(
    n_samples=600,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    class_sep=1.2,
    random_state=42,
)

feature_names = [f"x{i}" for i in range(X.shape[1])]
df_cls = pd.DataFrame(X, columns=feature_names)
df_cls["label"] = y

df_cls.head()


訓練用と評価用を分けるのは、まだ見ていないデータでも当たるかを確かめるためです。教師あり学習では、この分割を入れないと精度が良く見えすぎます。

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df_cls[feature_names],
    df_cls["label"],
    test_size=0.25,
    random_state=42,
    stratify=df_cls["label"],
)

clf = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
clf.fit(X_train, y_train)

pred = clf.predict(X_test)
acc = accuracy_score(y_test, pred)
print(f"test accuracy: {acc:.3f}")


分類では、accuracy だけでなく混同行列を見ると、どちらのクラスで間違っているかが具体的に見えます。正解率 1 つでは、誤り方の偏りが隠れてしまいます。

In [ ]:
cm = confusion_matrix(y_test, pred)
fig, ax = plt.subplots(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.show()

print(classification_report(y_test, pred, digits=3))


標準化の有無を比べると、前処理がモデルの性能にどう効くかを体感できます。教師あり学習は、モデルだけでなく前処理込みで設計するものだと分かるはずです。

In [ ]:
clf_no_scale = LogisticRegression(max_iter=1000, random_state=42)
clf_no_scale.fit(X_train, y_train)
pred_no_scale = clf_no_scale.predict(X_test)

acc_no_scale = accuracy_score(y_test, pred_no_scale)
print(f"without scaling: {acc_no_scale:.3f}")
print(f"with scaling   : {acc:.3f}")


## 教師なし学習: ラベルなしでまとまりを探す

次はクラスタリングです。ここでは学習時に正解ラベルを使いません。似た点を同じグループに寄せること自体が目的になります。

In [ ]:
X_blob, _ = make_blobs(
    n_samples=450,
    centers=3,
    cluster_std=1.2,
    n_features=2,
    random_state=42,
)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(X_blob[:, 0], X_blob[:, 1], s=18, alpha=0.8)
ax.set_title("Unlabeled Data for Clustering")
ax.set_xlabel("feature 1")
ax.set_ylabel("feature 2")
plt.show()


クラスタ数 `k` は先に与える必要があります。`inertia` と `silhouette` はその妥当さを見るための補助的な数字ですが、どちらも万能ではありません。ここでは「極端に不自然な k を避けるための目安」として読むのがよいです。

In [ ]:
rows = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = km.fit_predict(X_blob)
    rows.append({
        "k": k,
        "inertia": km.inertia_,
        "silhouette": silhouette_score(X_blob, labels)
    })

k_report = pd.DataFrame(rows)
k_report


ここでは `k=3` を選んで可視化します。教師あり学習と違って、評価は「正解と一致したか」ではなく、点群のまとまりや分離の良さとして読みます。

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=20, random_state=42)
cluster_id = kmeans.fit_predict(X_blob)
centers = kmeans.cluster_centers_

fig, ax = plt.subplots(figsize=(5.5, 4.2))
scatter = ax.scatter(X_blob[:, 0], X_blob[:, 1], c=cluster_id, cmap="tab10", s=22, alpha=0.8)
ax.scatter(centers[:, 0], centers[:, 1], c="black", marker="X", s=140, label="centers")
ax.legend(loc="best")
ax.set_title("KMeans Clustering (k=3)")
ax.set_xlabel("feature 1")
ax.set_ylabel("feature 2")
plt.show()


PCA は、元の特徴量を少数の軸へ写し直して全体像を見やすくする方法です。主成分は元の列の混ぜ合わせですが、ここではまず「情報をなるべく落とさずに 2 次元へ写している」と理解すれば十分です。

In [ ]:
iris = load_iris(as_frame=True)
X_iris = iris.data
y_iris = iris.target

X_iris_scaled = StandardScaler().fit_transform(X_iris)
pca = PCA(n_components=2, random_state=42)
X_iris_2d = pca.fit_transform(X_iris_scaled)

pca_df = pd.DataFrame(X_iris_2d, columns=["PC1", "PC2"])
pca_df["species"] = y_iris.map({0: "setosa", 1: "versicolor", 2: "virginica"})

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="species", ax=ax)
ax.set_title("PCA Projection of Iris")
plt.show()

print("explained variance ratio:", pca.explained_variance_ratio_)


## まとめ

教師あり学習は正解ラベルに向かって予測器を作る学習、教師なし学習はラベルなしで構造を探る学習です。実務では、教師なしでデータの形を把握してから、教師ありで予測問題を組むこともよくあります。違いを言葉で説明できることが第一歩です。